# 基于IMDB数据集的情感二分类实验

## 毕业设计实验代码

### 实验概述
本实验基于IMDB电影评论数据集，使用多种自然语言处理方法进行情感二分类（正面/负面），
系统性地对比传统机器学习方法与深度学习方法在文本情感分析任务上的表现。

### 实验模型
| 序号 | 模型 | 类别 | 核心思想 |
|------|------|------|----------|
| 1 | Naive Bayes + TF-IDF | 传统机器学习 | 基于贝叶斯定理的概率分类 |
| 2 | TextCNN | 深度学习 | 卷积神经网络提取局部n-gram特征 |
| 3 | BiLSTM | 深度学习 | 双向LSTM捕捉长距离上下文依赖 |
| 4 | BERT | 预训练模型微调 | Transformer双向编码器，迁移学习 |

### 评价指标
- Accuracy（准确率）
- Precision（精确率）
- Recall（召回率）
- F1-Score（F1分数）
- Confusion Matrix（混淆矩阵）
- ROC Curve & AUC（ROC曲线与AUC值）

### 数据集
- IMDB Large Movie Review Dataset (ACL 2011)
- 训练集：25,000条评论（正面12,500 + 负面12,500）
- 测试集：25,000条评论（正面12,500 + 负面12,500）

## 第1部分：环境配置与依赖导入

导入实验所需的全部Python库：
- **os, re**: 文件操作与正则表达式（用于数据加载和文本清洗）
- **numpy, pandas**: 数值计算与数据处理
- **matplotlib, seaborn**: 数据可视化
- **nltk**: 自然语言处理工具包（分词、停用词）
- **sklearn**: 传统机器学习（TF-IDF、朴素贝叶斯、评价指标）
- **torch**: PyTorch深度学习框架（TextCNN、BiLSTM）
- **transformers**: Hugging Face预训练模型库（BERT）

In [ ]:
# ============================================================================
# 第1部分：导入所有依赖库
# ============================================================================

# --- 基础工具库 ---
import os                          # 操作系统接口，用于文件路径操作
import re                          # 正则表达式，用于文本清洗
import warnings                    # 警告管理
import time                        # 计时
import numpy as np                 # 数值计算库
import pandas as pd                # 数据处理库（DataFrame）
import matplotlib.pyplot as plt    # 绑图库
import seaborn as sns              # 高级可视化库（基于matplotlib）
from collections import Counter    # 计数器，用于统计词频

# --- 设置matplotlib支持中文显示 ---
plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False  # 解决负号显示问题

# --- NLTK：自然语言处理工具包 ---
import nltk
from nltk.corpus import stopwords       # 停用词表（the, is, a 等无实际意义的词）
from nltk.tokenize import word_tokenize # 分词器

# --- Scikit-learn：传统机器学习库 ---
from sklearn.feature_extraction.text import TfidfVectorizer  # TF-IDF特征提取
from sklearn.naive_bayes import MultinomialNB                # 多项式朴素贝叶斯
from sklearn.metrics import (
    accuracy_score,           # 准确率
    precision_score,          # 精确率
    recall_score,             # 召回率
    f1_score,                 # F1分数
    confusion_matrix,         # 混淆矩阵
    classification_report,    # 分类报告
    roc_curve,                # ROC曲线
    auc                       # AUC值
)

# --- PyTorch：深度学习框架 ---
import torch
import torch.nn as nn              # 神经网络模块
import torch.optim as optim        # 优化器
import torch.utils.data as data    # 数据加载工具

# --- Transformers：Hugging Face 预训练模型库 ---
from transformers import BertTokenizer, BertForSequenceClassification

# --- 下载NLTK必要资源 ---
nltk.download('stopwords', quiet=True)   # 英文停用词表
nltk.download('punkt', quiet=True)       # 分词模型
nltk.download('punkt_tab', quiet=True)   # 分词模型（新版本需要）

# --- 加载英文停用词 ---
stop_words = set(stopwords.words('english'))

# --- 忽略不必要的警告 ---
warnings.filterwarnings('ignore')

print("所有依赖库导入成功！")

## 第2部分：全局超参数配置

统一管理实验中所有模型共用的超参数，方便后续调参和复现实验。

In [ ]:
# ============================================================================
# 第2部分：全局超参数配置
# ============================================================================
# 这些参数会被后续所有深度学习模型共用

MAX_VOCAB_SIZE = 10000   # 词表最大大小：只保留语料中最常见的10,000个词
                         # 过大会引入噪声词，过小会丢失语义信息

MAX_SEQ_LEN    = 200     # 输入序列最大长度：超过200个词则截断，不足则用0填充(padding)
                         # IMDB评论平均长度约230词，200可覆盖大部分内容

EMBEDDING_DIM  = 128     # 词向量维度：每个词用128维的稠密向量表示
                         # 维度越高表达能力越强，但计算量也越大

BATCH_SIZE     = 64      # 批大小：每次训练使用64个样本计算梯度
                         # 太大显存不够，太小训练不稳定

EPOCHS         = 5       # 训练轮数：整个训练集遍历5次
                         # TextCNN和BiLSTM使用5轮，BERT使用2-3轮

LEARNING_RATE  = 1e-3    # 学习率：TextCNN和BiLSTM使用0.001
                         # BERT微调使用更小的学习率2e-5

BERT_LR        = 2e-5    # BERT专用学习率：预训练模型微调需要很小的学习率
                         # 避免破坏预训练阶段学到的语言知识

BERT_MAX_LEN   = 128     # BERT输入最大长度：BERT使用WordPiece分词，128足够

BERT_EPOCHS    = 3       # BERT训练轮数：预训练模型通常2-3轮即可收敛

# --- 自动检测计算设备 ---
# 优先使用GPU（CUDA），如果没有GPU则使用CPU
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'计算设备: {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'GPU型号: {torch.cuda.get_device_name(0)}')
    print(f'GPU显存: {torch.cuda.get_device_properties(0).total_mem / 1024**3:.1f} GB')

## 第3部分：数据加载与文本预处理

### 3.1 数据加载
IMDB数据集的目录结构如下：
```
aclImdb/
├── train/
│   ├── pos/   (12,500条正面评论，标签=1)
│   └── neg/   (12,500条负面评论，标签=0)
└── test/
    ├── pos/   (12,500条正面评论)
    └── neg/   (12,500条负面评论)
```

### 3.2 文本清洗流程
1. 去除HTML标签（如 `<br>`, `<p>` 等）
2. 去除非字母字符（数字、标点等）
3. 转换为小写
4. NLTK分词
5. 去除英文停用词（the, is, a, an 等无实际语义的高频词）

In [ ]:
# ============================================================================
# 第3部分：数据加载与文本预处理
# ============================================================================

def load_imdb_data(data_dir='aclImdb'):
    """
    加载IMDB电影评论数据集
    
    数据集来源: http://ai.stanford.edu/~amaas/data/sentiment/
    每条评论存储为一个独立的.txt文件，文件名格式: id_rating.txt
    
    参数:
        data_dir (str): 数据集根目录路径
    
    返回:
        train_df (DataFrame): 训练集，包含 'text' 和 'label' 两列
        test_df  (DataFrame): 测试集，包含 'text' 和 'label' 两列
    """
    
    def load_data_from_dir(dir_path, label):
        """
        从指定目录读取所有.txt文件
        
        参数:
            dir_path (str): 目录路径（如 aclImdb/train/pos）
            label    (int): 标签值（1=正面, 0=负面）
        
        返回:
            texts  (list): 文本列表
            labels (list): 标签列表
        """
        texts, labels = [], []
        for filename in os.listdir(dir_path):
            if filename.endswith('.txt'):
                filepath = os.path.join(dir_path, filename)
                with open(filepath, 'r', encoding='utf-8') as f:
                    texts.append(f.read())
                    labels.append(label)
        return texts, labels
    
    # --- 加载训练集 ---
    # pos目录: 正面评论，标签为1
    # neg目录: 负面评论，标签为0
    print("  加载训练集正面评论...")
    train_pos, train_pos_labels = load_data_from_dir(
        os.path.join(data_dir, 'train', 'pos'), 1
    )
    print("  加载训练集负面评论...")
    train_neg, train_neg_labels = load_data_from_dir(
        os.path.join(data_dir, 'train', 'neg'), 0
    )
    
    # --- 加载测试集 ---
    print("  加载测试集正面评论...")
    test_pos, test_pos_labels = load_data_from_dir(
        os.path.join(data_dir, 'test', 'pos'), 1
    )
    print("  加载测试集负面评论...")
    test_neg, test_neg_labels = load_data_from_dir(
        os.path.join(data_dir, 'test', 'neg'), 0
    )
    
    # --- 合并为DataFrame ---
    # 将正面和负面评论合并到一起
    train_df = pd.DataFrame({
        'text':  train_pos + train_neg,
        'label': train_pos_labels + train_neg_labels
    })
    test_df = pd.DataFrame({
        'text':  test_pos + test_neg,
        'label': test_pos_labels + test_neg_labels
    })
    
    # --- 随机打乱顺序 ---
    # random_state=42 保证每次运行结果一致（可复现）
    train_df = train_df.sample(frac=1, random_state=42).reset_index(drop=True)
    test_df  = test_df.sample(frac=1, random_state=42).reset_index(drop=True)
    
    return train_df, test_df


def clean_text(text):
    """
    文本清洗函数
    
    对原始评论文本进行预处理，去除噪声信息，保留有意义的词汇。
    
    处理流程:
        原始文本 -> 去HTML标签 -> 去特殊字符 -> 小写化 -> 分词 -> 去停用词 -> 拼接
    
    参数:
        text (str): 原始评论文本
    
    返回:
        str: 清洗后的文本
    
    示例:
        输入: "<br>This is a GREAT movie! I loved it 100%.<br>"
        输出: "great movie loved"
    """
    # 步骤1: 去除HTML标签（IMDB评论中包含大量<br>等标签）
    text = re.sub(r'<.*?>', '', text)
    
    # 步骤2: 只保留英文字母和空格，去除数字和标点符号
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    
    # 步骤3: 统一转为小写（"Good"和"good"应视为同一个词）
    text = text.lower()
    
    # 步骤4: 使用NLTK进行分词（比简单的split更准确）
    tokens = word_tokenize(text)
    
    # 步骤5: 去除停用词（the, is, a, an, in 等高频但无语义的词）
    tokens = [t for t in tokens if t not in stop_words]
    
    # 步骤6: 将词列表重新拼接为字符串
    return ' '.join(tokens)


# --- 执行数据加载 ---
print("=" * 50)
print("开始加载IMDB数据集...")
print("=" * 50)
start_time = time.time()

train_df, test_df = load_imdb_data()

print(f"\n数据加载完成！耗时: {time.time() - start_time:.1f}秒")
print(f"训练集: {len(train_df)} 条评论")
print(f"测试集: {len(test_df)} 条评论")

# --- 执行文本清洗 ---
print("\n开始文本清洗（这一步需要一些时间）...")
start_time = time.time()

train_df['clean_text'] = train_df['text'].apply(clean_text)
test_df['clean_text']  = test_df['text'].apply(clean_text)

print(f"文本清洗完成！耗时: {time.time() - start_time:.1f}秒")

# --- 提取标签数组 ---
y_train = train_df['label'].values  # 训练集标签 (numpy数组)
y_test  = test_df['label'].values   # 测试集标签 (numpy数组)

# --- 展示清洗效果 ---
print("\n--- 清洗前后对比示例 ---")
idx = 0
print(f"原始文本（前200字符）:\n{train_df['text'].iloc[idx][:200]}...")
print(f"\n清洗后文本（前200字符）:\n{train_df['clean_text'].iloc[idx][:200]}...")
print(f"\n标签: {'正面' if train_df['label'].iloc[idx] == 1 else '负面'}")

## 第4部分：探索性数据分析（EDA）

在训练模型之前，先对数据集进行全面的统计分析和可视化，了解数据的分布特征：
1. 标签分布（正面/负面比例）
2. 文本长度分布
3. 高频词统计（词云）
4. 正负面评论的词汇差异

In [ ]:
# ============================================================================
# 第4部分：探索性数据分析（EDA）
# ============================================================================

# --- 4.1 标签分布分析 ---
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# 训练集标签分布
train_label_counts = train_df['label'].value_counts()
axes[0].bar(['负面 (0)', '正面 (1)'], train_label_counts.values,
            color=['#e74c3c', '#2ecc71'], edgecolor='black')
axes[0].set_title('训练集标签分布', fontsize=14)
axes[0].set_ylabel('样本数量')
for i, v in enumerate(train_label_counts.values):
    axes[0].text(i, v + 100, str(v), ha='center', fontsize=12, fontweight='bold')

# 测试集标签分布
test_label_counts = test_df['label'].value_counts()
axes[1].bar(['负面 (0)', '正面 (1)'], test_label_counts.values,
            color=['#e74c3c', '#2ecc71'], edgecolor='black')
axes[1].set_title('测试集标签分布', fontsize=14)
axes[1].set_ylabel('样本数量')
for i, v in enumerate(test_label_counts.values):
    axes[1].text(i, v + 100, str(v), ha='center', fontsize=12, fontweight='bold')

plt.suptitle('数据集标签分布（正面 vs 负面）', fontsize=16, y=1.02)
plt.tight_layout()
plt.savefig('eda_label_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"训练集 - 正面: {train_label_counts.get(1, 0)}, 负面: {train_label_counts.get(0, 0)}")
print(f"测试集 - 正面: {test_label_counts.get(1, 0)}, 负面: {test_label_counts.get(0, 0)}")
print("数据集正负样本均衡，无需进行过采样/欠采样处理。")

In [ ]:
# --- 4.2 文本长度分布分析 ---
# 计算每条评论清洗后的词数
train_df['text_length'] = train_df['clean_text'].apply(lambda x: len(x.split()))
test_df['text_length']  = test_df['clean_text'].apply(lambda x: len(x.split()))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 训练集文本长度分布直方图
axes[0].hist(train_df['text_length'], bins=50, color='#3498db', alpha=0.7, edgecolor='black')
axes[0].axvline(x=train_df['text_length'].mean(), color='red', linestyle='--',
                label=f"均值: {train_df['text_length'].mean():.0f}")
axes[0].axvline(x=train_df['text_length'].median(), color='orange', linestyle='--',
                label=f"中位数: {train_df['text_length'].median():.0f}")
axes[0].axvline(x=MAX_SEQ_LEN, color='green', linestyle='--',
                label=f"截断长度: {MAX_SEQ_LEN}")
axes[0].set_title('训练集文本长度分布', fontsize=14)
axes[0].set_xlabel('词数')
axes[0].set_ylabel('评论数量')
axes[0].legend()

# 正面 vs 负面 文本长度对比（箱线图）
train_df.boxplot(column='text_length', by='label', ax=axes[1],
                 patch_artist=True,
                 boxprops=dict(facecolor='#3498db', alpha=0.5))
axes[1].set_title('正面 vs 负面评论长度对比', fontsize=14)
axes[1].set_xlabel('标签 (0=负面, 1=正面)')
axes[1].set_ylabel('词数')
plt.suptitle('')  # 去掉自动生成的标题

plt.tight_layout()
plt.savefig('eda_text_length.png', dpi=150, bbox_inches='tight')
plt.show()

# 打印统计信息
print("--- 文本长度统计 ---")
print(f"训练集: 均值={train_df['text_length'].mean():.0f}, "
      f"中位数={train_df['text_length'].median():.0f}, "
      f"最大={train_df['text_length'].max()}, "
      f"最小={train_df['text_length'].min()}")
print(f"超过{MAX_SEQ_LEN}词的评论占比: "
      f"{(train_df['text_length'] > MAX_SEQ_LEN).mean() * 100:.1f}%")

In [ ]:
# --- 4.3 高频词分析 ---
# 分别统计正面和负面评论中的高频词

# 正面评论词频统计
pos_words = Counter()
for text in train_df[train_df['label'] == 1]['clean_text']:
    pos_words.update(text.split())

# 负面评论词频统计
neg_words = Counter()
for text in train_df[train_df['label'] == 0]['clean_text']:
    neg_words.update(text.split())

# 取Top20高频词
top_n = 20
pos_top = pos_words.most_common(top_n)
neg_top = neg_words.most_common(top_n)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# 正面评论高频词
pos_words_list, pos_counts = zip(*pos_top)
axes[0].barh(range(top_n), pos_counts[::-1], color='#2ecc71', edgecolor='black')
axes[0].set_yticks(range(top_n))
axes[0].set_yticklabels(pos_words_list[::-1])
axes[0].set_title(f'正面评论 Top{top_n} 高频词', fontsize=14)
axes[0].set_xlabel('出现次数')

# 负面评论高频词
neg_words_list, neg_counts = zip(*neg_top)
axes[1].barh(range(top_n), neg_counts[::-1], color='#e74c3c', edgecolor='black')
axes[1].set_yticks(range(top_n))
axes[1].set_yticklabels(neg_words_list[::-1])
axes[1].set_title(f'负面评论 Top{top_n} 高频词', fontsize=14)
axes[1].set_xlabel('出现次数')

plt.suptitle('正面 vs 负面评论高频词对比', fontsize=16, y=1.02)
plt.tight_layout()
plt.savefig('eda_top_words.png', dpi=150, bbox_inches='tight')
plt.show()

# --- 4.4 正负面评论的差异词分析 ---
# 找出在正面评论中高频但在负面评论中低频的词（反之亦然）
# 使用词频比值来衡量
print("\n--- 正面评论特征词（正面频率/负面频率 最高的词）---")
pos_ratio = {}
for word, count in pos_words.most_common(500):
    neg_count = neg_words.get(word, 1)  # 避免除以0
    pos_ratio[word] = count / neg_count

pos_distinctive = sorted(pos_ratio.items(), key=lambda x: x[1], reverse=True)[:15]
for word, ratio in pos_distinctive:
    print(f"  {word:15s} 正面:{pos_words[word]:5d}  负面:{neg_words.get(word,0):5d}  比值:{ratio:.2f}")

print("\n--- 负面评论特征词（负面频率/正面频率 最高的词）---")
neg_ratio = {}
for word, count in neg_words.most_common(500):
    pos_count = pos_words.get(word, 1)
    neg_ratio[word] = count / pos_count

neg_distinctive = sorted(neg_ratio.items(), key=lambda x: x[1], reverse=True)[:15]
for word, ratio in neg_distinctive:
    print(f"  {word:15s} 负面:{neg_words[word]:5d}  正面:{pos_words.get(word,0):5d}  比值:{ratio:.2f}")

## 第5部分：模型1 - 朴素贝叶斯 (Naive Bayes + TF-IDF)

### 算法原理
- **贝叶斯定理**: P(类别|特征) ∝ P(特征|类别) × P(类别)
- **"朴素"假设**: 各特征（词）之间相互独立
- **TF-IDF特征**: 
  - TF（词频）= 词在文档中出现的次数 / 文档总词数
  - IDF（逆文档频率）= log(总文档数 / 包含该词的文档数)
  - TF-IDF = TF × IDF，衡量词对文档的重要程度

### 优化策略
- 使用 bigram (ngram_range=(1,2)) 捕捉短语特征
- sublinear_tf=True 平滑高频词影响
- 调低拉普拉斯平滑系数 alpha=0.1

In [ ]:
# ============================================================================
# 第5部分：朴素贝叶斯模型 (Naive Bayes + TF-IDF)
# ============================================================================
print("=" * 50)
print("训练模型1：朴素贝叶斯 (Naive Bayes)")
print("=" * 50)

# --- 5.1 TF-IDF 特征提取 ---
# TF-IDF (Term Frequency - Inverse Document Frequency) 将文本转为数值特征向量
# 核心思想：一个词在某文档中出现频率高（TF高），但在整个语料中出现少（IDF高），
#           则该词对该文档的区分度高，TF-IDF值大
tfidf = TfidfVectorizer(
    max_features=MAX_VOCAB_SIZE,  # 只保留最重要的10000个特征词
    ngram_range=(1, 2),           # 同时使用 unigram 和 bigram
                                  # unigram: "good", "movie"
                                  # bigram:  "good movie", "not good"
                                  # bigram能捕捉否定等短语特征
    min_df=5,                     # 过滤掉在少于5篇文档中出现的词（去除罕见词）
    max_df=0.8,                   # 过滤掉在超过80%文档中出现的词（去除过于常见的词）
    sublinear_tf=True             # 使用 1 + log(tf) 代替 tf
                                  # 避免某个词出现100次比出现1次重要100倍的问题
)

# fit_transform: 在训练集上拟合（学习词表和IDF值）并转换
X_train_tfidf = tfidf.fit_transform(train_df['clean_text'])
# transform: 在测试集上只转换（使用训练集学到的词表和IDF值）
# 注意：绝不能在测试集上fit，否则会造成数据泄露
X_test_tfidf = tfidf.transform(test_df['clean_text'])

print(f"TF-IDF特征矩阵维度: 训练集{X_train_tfidf.shape}, 测试集{X_test_tfidf.shape}")

# --- 5.2 训练朴素贝叶斯模型 ---
# MultinomialNB: 多项式朴素贝叶斯，适用于离散特征（如词频、TF-IDF）
# alpha: 拉普拉斯平滑系数，防止某个特征概率为0导致整体概率为0
#        alpha越小，模型对训练数据越敏感
nb_model = MultinomialNB(alpha=0.1)
nb_model.fit(X_train_tfidf, y_train)

# --- 5.3 预测与评估 ---
y_pred_nb = nb_model.predict(X_test_tfidf)           # 预测类别标签
y_prob_nb = nb_model.predict_proba(X_test_tfidf)[:, 1]  # 预测为正面的概率（用于ROC曲线）

# 计算各项评价指标
nb_acc       = accuracy_score(y_test, y_pred_nb)
nb_precision = precision_score(y_test, y_pred_nb)
nb_recall    = recall_score(y_test, y_pred_nb)
nb_f1        = f1_score(y_test, y_pred_nb)

print(f"\n--- 朴素贝叶斯评估结果 ---")
print(f"Accuracy  (准确率): {nb_acc:.4f}")
print(f"Precision (精确率): {nb_precision:.4f}")
print(f"Recall    (召回率): {nb_recall:.4f}")
print(f"F1 Score  (F1分数): {nb_f1:.4f}")
print(f"\n详细分类报告:")
print(classification_report(y_test, y_pred_nb, target_names=['负面', '正面']))

# --- 5.4 混淆矩阵可视化 ---
cm_nb = confusion_matrix(y_test, y_pred_nb)
plt.figure(figsize=(6, 5))
sns.heatmap(cm_nb, annot=True, fmt='d', cmap='Blues',
            xticklabels=['负面', '正面'], yticklabels=['负面', '正面'])
plt.xlabel('预测标签', fontsize=12)
plt.ylabel('真实标签', fontsize=12)
plt.title('朴素贝叶斯 - 混淆矩阵', fontsize=14)
plt.tight_layout()
plt.savefig('nb_confusion_matrix.png', dpi=150)
plt.show()

## 第6部分：深度学习数据准备（词表构建与序列编码）

深度学习模型无法直接处理文本字符串，需要将文本转换为数字序列：

```
原始文本 → 分词 → 查词表 → 整数序列 → Embedding向量
"good movie" → ["good", "movie"] → [42, 156] → [[0.1, 0.3, ...], [0.5, 0.2, ...]]
```

关键概念：
- **词表 (Vocabulary)**: 词到整数索引的映射字典
- **`<pad>` (索引=0)**: 填充符，将短序列填充到统一长度
- **`<unk>` (索引=1)**: 未知词符，表示不在词表中的词
- **Embedding**: 将离散的整数索引映射为连续的稠密向量

In [ ]:
# ============================================================================
# 第6部分：深度学习数据准备 - 词表构建与序列编码
# ============================================================================

# --- 6.1 构建词表 ---
def build_vocab(texts, max_tokens):
    """
    从训练语料中构建词表（word → index 映射）
    
    流程:
        1. 统计所有词的出现频率
        2. 按频率降序排列
        3. 取前 max_tokens-2 个词（预留<pad>和<unk>的位置）
        4. 为每个词分配唯一的整数索引
    
    参数:
        texts (Series/list): 清洗后的文本列表
        max_tokens (int):    词表最大大小
    
    返回:
        word2idx (dict): 词到索引的映射字典
                         例如: {'<pad>': 0, '<unk>': 1, 'movie': 2, 'good': 3, ...}
    """
    # 统计所有词的出现次数
    counter = Counter()
    for text in texts:
        counter.update(text.split())
    
    # 取出现频率最高的词（预留2个位置给特殊token）
    most_common = counter.most_common(max_tokens - 2)
    
    # 构建映射字典
    # <pad>=0: 填充符，用于将短序列填充到固定长度
    # <unk>=1: 未知词符，用于表示词表外的词（Out-of-Vocabulary, OOV）
    word2idx = {'<pad>': 0, '<unk>': 1}
    for word, _ in most_common:
        word2idx[word] = len(word2idx)
    
    return word2idx


# 在训练集上构建词表（注意：只用训练集，不用测试集，避免数据泄露）
vocab = build_vocab(train_df['clean_text'], MAX_VOCAB_SIZE)
PAD_IDX = vocab['<pad>']  # 填充符索引 = 0
UNK_IDX = vocab['<unk>']  # 未知词索引 = 1
VOCAB_SIZE = len(vocab)    # 实际词表大小

print(f"词表大小: {VOCAB_SIZE}")
print(f"词表示例（前10个词）: {list(vocab.items())[:10]}")


# --- 6.2 文本编码函数 ---
def encode_text(text):
    """
    将清洗后的文本编码为固定长度的整数序列
    
    流程:
        1. 分词
        2. 查词表获取索引（未知词用 <unk> 的索引替代）
        3. 截断或填充到 MAX_SEQ_LEN 长度
    
    参数:
        text (str): 清洗后的文本
    
    返回:
        ids (list): 长度为 MAX_SEQ_LEN 的整数索引列表
    
    示例:
        输入: "good movie great"
        输出: [42, 156, 78, 0, 0, 0, ..., 0]  (长度=MAX_SEQ_LEN)
    """
    tokens = text.split()
    # 查词表：如果词在词表中则返回对应索引，否则返回 <unk> 的索引
    ids = [vocab.get(token, UNK_IDX) for token in tokens]
    
    # 填充或截断到固定长度
    if len(ids) < MAX_SEQ_LEN:
        # 短序列：在末尾填充 <pad>（索引=0）
        ids = ids + [PAD_IDX] * (MAX_SEQ_LEN - len(ids))
    else:
        # 长序列：截断到 MAX_SEQ_LEN
        ids = ids[:MAX_SEQ_LEN]
    
    return ids


# --- 6.3 编码所有数据 ---
print("\n编码训练集和测试集文本...")
start_time = time.time()

train_ids = np.array([encode_text(t) for t in train_df['clean_text']])
test_ids  = np.array([encode_text(t) for t in test_df['clean_text']])

print(f"编码完成！耗时: {time.time() - start_time:.1f}秒")
print(f"训练集编码矩阵: {train_ids.shape}")  # (25000, 200)
print(f"测试集编码矩阵: {test_ids.shape}")    # (25000, 200)


# --- 6.4 创建 PyTorch DataLoader ---
# TensorDataset: 将特征和标签打包为数据集
train_dataset = data.TensorDataset(
    torch.tensor(train_ids, dtype=torch.long),   # 输入序列 (int64)
    torch.tensor(y_train, dtype=torch.long)       # 标签 (int64)
)
test_dataset = data.TensorDataset(
    torch.tensor(test_ids, dtype=torch.long),
    torch.tensor(y_test, dtype=torch.long)
)

# DataLoader: 自动分批、打乱数据
# shuffle=True: 训练时每个epoch随机打乱数据顺序，防止模型记住数据顺序
# 测试集不需要打乱
train_loader = data.DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader  = data.DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f"\nDataLoader创建完成:")
print(f"训练集: {len(train_loader)} 个batch (每batch {BATCH_SIZE} 条)")
print(f"测试集: {len(test_loader)} 个batch")

## 第7部分：通用训练与评估函数

定义深度学习模型共用的训练循环和评估函数，避免代码重复。

In [ ]:
# ============================================================================
# 第7部分：通用训练与评估函数
# ============================================================================

def train_epoch(model, loader, criterion, optimizer):
    """
    训练模型一个epoch（遍历整个训练集一次）
    
    训练流程（每个batch）:
        1. 前向传播：输入数据 → 模型 → 预测值
        2. 计算损失：预测值 vs 真实标签 → loss
        3. 反向传播：计算每个参数的梯度 (∂loss/∂w)
        4. 参数更新：w = w - lr × ∂loss/∂w
    
    参数:
        model:     PyTorch模型
        loader:    DataLoader（训练数据）
        criterion: 损失函数（如BCELoss）
        optimizer: 优化器（如Adam）
    
    返回:
        avg_loss (float): 该epoch的平均损失值
    """
    model.train()  # 切换到训练模式（启用Dropout、BatchNorm等）
    total_loss = 0
    
    for xb, yb in loader:
        # 将数据移到GPU（如果可用）
        xb = xb.to(DEVICE)
        yb = yb.float().to(DEVICE)  # BCELoss需要float类型标签
        
        optimizer.zero_grad()        # 步骤0: 清零上一步的梯度（PyTorch梯度会累加）
        preds = model(xb)            # 步骤1: 前向传播，得到预测概率
        loss = criterion(preds, yb)  # 步骤2: 计算损失
        loss.backward()              # 步骤3: 反向传播，计算梯度
        optimizer.step()             # 步骤4: 更新参数
        
        total_loss += loss.item()    # 累加损失值（.item()将tensor转为Python数值）
    
    return total_loss / len(loader)  # 返回平均损失


def eval_model(model, loader):
    """
    评估模型在测试集上的表现
    
    参数:
        model:  PyTorch模型
        loader: DataLoader（测试数据）
    
    返回:
        acc (float):   准确率
        f1 (float):    F1分数
        preds (list):  预测的概率值（用于后续ROC曲线绘制）
        labels (list): 真实标签
    """
    model.eval()  # 切换到评估模式（关闭Dropout、BatchNorm使用全局统计量）
    all_preds, all_labels = [], []
    
    with torch.no_grad():  # 评估时不需要计算梯度，节省显存和计算
        for xb, yb in loader:
            xb = xb.to(DEVICE)
            preds = model(xb).cpu().numpy()  # 预测并移回CPU
            all_preds.extend(preds)
            all_labels.extend(yb.numpy())
    
    # 将概率转为类别标签（阈值=0.5）
    preds_bin = (np.array(all_preds) > 0.5).astype(int)
    
    acc = accuracy_score(all_labels, preds_bin)
    f1  = f1_score(all_labels, preds_bin)
    
    return acc, f1, np.array(all_preds), np.array(all_labels)


def train_and_evaluate(model, model_name, train_loader, test_loader, 
                       criterion, optimizer, epochs):
    """
    完整的训练+评估流程，记录每个epoch的指标
    
    参数:
        model:        PyTorch模型
        model_name:   模型名称（用于打印）
        train_loader: 训练数据DataLoader
        test_loader:  测试数据DataLoader
        criterion:    损失函数
        optimizer:    优化器
        epochs:       训练轮数
    
    返回:
        history (dict): 训练历史记录
            - 'train_loss': 每个epoch的训练损失
            - 'test_acc':   每个epoch的测试准确率
            - 'test_f1':    每个epoch的测试F1分数
    """
    history = {'train_loss': [], 'test_acc': [], 'test_f1': []}
    
    print(f"\n{'='*50}")
    print(f"开始训练 {model_name}")
    print(f"{'='*50}")
    
    for epoch in range(epochs):
        start_time = time.time()
        
        # 训练一个epoch
        train_loss = train_epoch(model, train_loader, criterion, optimizer)
        
        # 在测试集上评估
        test_acc, test_f1, _, _ = eval_model(model, test_loader)
        
        elapsed = time.time() - start_time
        
        # 记录历史
        history['train_loss'].append(train_loss)
        history['test_acc'].append(test_acc)
        history['test_f1'].append(test_f1)
        
        print(f"Epoch {epoch+1}/{epochs} - "
              f"Loss: {train_loss:.4f} - "
              f"Acc: {test_acc:.4f} - "
              f"F1: {test_f1:.4f} - "
              f"耗时: {elapsed:.1f}s")
    
    return history


print("通用训练/评估函数定义完成。")

## 第8部分：模型2 - TextCNN（文本卷积神经网络）

### 算法原理
TextCNN 由 Yoon Kim 在 2014 年提出，将计算机视觉中的 CNN 思想应用于文本分类。

### 核心思想
- 使用不同大小的卷积核（kernel_size=3,4,5）提取不同长度的 n-gram 特征
- kernel_size=3 → 捕捉3个连续词的组合（如 "not very good"）
- 全局最大池化（Global Max Pooling）提取每个卷积核最显著的特征

### 网络结构
```
Input(序列) → Embedding → Conv1D(k=3) → ReLU → MaxPool
                        → Conv1D(k=4) → ReLU → MaxPool  → Concat → Dropout → FC → Sigmoid
                        → Conv1D(k=5) → ReLU → MaxPool
```

### 优缺点
- 优点：并行计算，训练速度快；能有效捕捉局部特征
- 缺点：难以捕捉长距离依赖关系

In [ ]:
# ============================================================================
# 第8部分：TextCNN 模型定义与训练
# ============================================================================

class TextCNN(nn.Module):
    """
    TextCNN 文本分类模型 (Yoon Kim, 2014)
    
    使用多个不同大小的卷积核并行提取文本的局部n-gram特征，
    然后通过全局最大池化和全连接层进行分类。
    
    参数:
        vocab_size (int):  词表大小
        embed_dim (int):   词向量维度
        num_filters (int): 每种卷积核的数量
        filter_sizes (list): 卷积核大小列表，如[3,4,5]
        num_classes (int): 输出类别数（二分类=1，输出概率）
    """
    def __init__(self, vocab_size, embed_dim, num_filters=64, 
                 filter_sizes=[3, 4, 5], num_classes=1):
        super().__init__()
        
        # --- Embedding层 ---
        # 将词索引映射为稠密向量
        # padding_idx=PAD_IDX: <pad>对应的向量始终为全0，不参与梯度更新
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=PAD_IDX)
        
        # --- 多尺度卷积层 ---
        # 使用ModuleList管理多个卷积层，每个卷积核大小不同
        # kernel_size=3: 每次看3个连续词 → 捕捉三元组特征 (trigram)
        # kernel_size=4: 每次看4个连续词 → 捕捉四元组特征
        # kernel_size=5: 每次看5个连续词 → 捕捉五元组特征
        self.convs = nn.ModuleList([
            nn.Conv1d(
                in_channels=embed_dim,    # 输入通道数 = 词向量维度
                out_channels=num_filters, # 输出通道数 = 卷积核数量
                kernel_size=fs            # 卷积核大小
            )
            for fs in filter_sizes
        ])
        
        # --- 全连接分类层 ---
        # 输入维度 = 卷积核数量 × 卷积核种类数
        self.fc = nn.Linear(num_filters * len(filter_sizes), num_classes)
        
        # --- Dropout层 ---
        # 训练时随机丢弃50%的神经元，防止过拟合
        self.dropout = nn.Dropout(0.5)
        
        # --- Sigmoid激活 ---
        # 将输出映射到(0,1)区间，表示正面评论的概率
        self.sigmoid = nn.Sigmoid()
    
    def forward(self, x):
        """
        前向传播
        
        数据流:
            x: (batch_size, seq_len)                    # 输入：整数索引序列
            → embedding: (batch_size, seq_len, embed_dim) # Embedding查表
            → permute: (batch_size, embed_dim, seq_len)   # 转置，适配Conv1d输入格式
            → conv+relu+maxpool: (batch_size, num_filters) × 3  # 多尺度卷积
            → concat: (batch_size, num_filters × 3)       # 拼接所有卷积结果
            → dropout → fc → sigmoid: (batch_size,)       # 分类输出概率
        """
        # Embedding: (batch, seq_len) → (batch, seq_len, embed_dim)
        x = self.embedding(x)
        
        # 转置: Conv1d期望输入格式为 (batch, channels, length)
        # (batch, seq_len, embed_dim) → (batch, embed_dim, seq_len)
        x = x.permute(0, 2, 1)
        
        # 对每个卷积核分别进行: 卷积 → ReLU激活 → 全局最大池化
        conv_outputs = []
        for conv in self.convs:
            # conv: (batch, embed_dim, seq_len) → (batch, num_filters, new_len)
            c = torch.relu(conv(x))
            # 全局最大池化: 取每个通道（filter）的最大值
            # (batch, num_filters, new_len) → (batch, num_filters)
            c = torch.max(c, dim=2)[0]
            conv_outputs.append(c)
        
        # 拼接所有卷积核的输出
        # [(batch, num_filters)] × 3 → (batch, num_filters × 3)
        x = torch.cat(conv_outputs, dim=1)
        
        # Dropout + 全连接 + Sigmoid
        x = self.dropout(x)
        x = self.fc(x)
        return self.sigmoid(x).squeeze(1)  # (batch,) 输出概率


# --- 初始化TextCNN模型 ---
model_cnn = TextCNN(
    vocab_size=VOCAB_SIZE,
    embed_dim=EMBEDDING_DIM,
    num_filters=64,
    filter_sizes=[3, 4, 5]
).to(DEVICE)  # 将模型移到GPU

# 损失函数: 二元交叉熵 (Binary Cross Entropy)
# 适用于二分类问题，衡量预测概率与真实标签之间的差异
criterion_cnn = nn.BCELoss()

# 优化器: Adam (Adaptive Moment Estimation)
# 自适应学习率优化器，结合了Momentum和RMSProp的优点
optimizer_cnn = optim.Adam(model_cnn.parameters(), lr=LEARNING_RATE)

# 打印模型结构
print("TextCNN 模型结构:")
print(model_cnn)
total_params = sum(p.numel() for p in model_cnn.parameters())
print(f"\n总参数量: {total_params:,}")

# --- 训练TextCNN ---
history_cnn = train_and_evaluate(
    model_cnn, "TextCNN", train_loader, test_loader,
    criterion_cnn, optimizer_cnn, EPOCHS
)

# --- 最终评估 ---
cnn_acc, cnn_f1, cnn_probs, cnn_labels = eval_model(model_cnn, test_loader)
cnn_preds = (cnn_probs > 0.5).astype(int)
cnn_precision = precision_score(cnn_labels, cnn_preds)
cnn_recall = recall_score(cnn_labels, cnn_preds)

print(f"\n--- TextCNN 最终评估结果 ---")
print(f"Accuracy:  {cnn_acc:.4f}")
print(f"Precision: {cnn_precision:.4f}")
print(f"Recall:    {cnn_recall:.4f}")
print(f"F1 Score:  {cnn_f1:.4f}")
print(classification_report(cnn_labels, cnn_preds, target_names=['负面', '正面']))

In [ ]:
# --- TextCNN 训练过程可视化 ---
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

epochs_range = range(1, EPOCHS + 1)

# 训练损失曲线
axes[0].plot(epochs_range, history_cnn['train_loss'], 'b-o', linewidth=2, markersize=6)
axes[0].set_title('TextCNN 训练损失曲线', fontsize=14)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].grid(True, alpha=0.3)

# 测试准确率曲线
axes[1].plot(epochs_range, history_cnn['test_acc'], 'g-o', linewidth=2, markersize=6)
axes[1].set_title('TextCNN 测试准确率曲线', fontsize=14)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].grid(True, alpha=0.3)

# 混淆矩阵
cm_cnn = confusion_matrix(cnn_labels, cnn_preds)
sns.heatmap(cm_cnn, annot=True, fmt='d', cmap='Greens', ax=axes[2],
            xticklabels=['负面', '正面'], yticklabels=['负面', '正面'])
axes[2].set_title('TextCNN 混淆矩阵', fontsize=14)
axes[2].set_xlabel('预测标签')
axes[2].set_ylabel('真实标签')

plt.tight_layout()
plt.savefig('textcnn_results.png', dpi=150, bbox_inches='tight')
plt.show()

## 第9部分：模型3 - BiLSTM（双向长短期记忆网络）

### 算法原理
LSTM (Long Short-Term Memory) 是一种特殊的RNN，通过门控机制解决了传统RNN的梯度消失问题。

### 三个门控机制
- **遗忘门 (Forget Gate)**: 决定丢弃哪些旧信息
- **输入门 (Input Gate)**: 决定存储哪些新信息
- **输出门 (Output Gate)**: 决定输出哪些信息

### 双向 (Bidirectional) 的优势
- 前向LSTM：从左到右阅读 → 捕捉"前文"信息
- 后向LSTM：从右到左阅读 → 捕捉"后文"信息
- 拼接两个方向的隐状态 → 获得更完整的上下文表示

### 网络结构
```
Input → Embedding → BiLSTM → 取最后时间步隐状态 → Dropout → FC → Sigmoid
```

### 优缺点
- 优点：能捕捉长距离依赖和双向上下文信息
- 缺点：序列处理无法并行，训练速度较慢

In [ ]:
# ============================================================================
# 第9部分：BiLSTM 模型定义与训练
# ============================================================================

class BiLSTM(nn.Module):
    """
    BiLSTM 双向长短期记忆网络文本分类模型
    
    使用双向LSTM处理文本序列，取最后时间步的隐状态作为文本表示，
    然后通过全连接层进行分类。
    
    参数:
        vocab_size (int):  词表大小
        embed_dim (int):   词向量维度
        hidden_dim (int):  LSTM隐状态维度
        num_layers (int):  LSTM层数
        num_classes (int): 输出类别数
    """
    def __init__(self, vocab_size, embed_dim, hidden_dim=64, 
                 num_layers=1, num_classes=1):
        super().__init__()
        
        # --- Embedding层 ---
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=PAD_IDX)
        
        # --- BiLSTM层 ---
        # input_size:    输入特征维度 = 词向量维度
        # hidden_size:   隐状态维度（每个方向）
        # num_layers:    LSTM堆叠层数
        # batch_first:   输入格式为 (batch, seq, feature)
        # bidirectional: 双向LSTM
        # dropout:       多层LSTM之间的Dropout（仅num_layers>1时生效）
        self.lstm = nn.LSTM(
            input_size=embed_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True,       # 双向：同时从左到右和从右到左处理
            dropout=0.3 if num_layers > 1 else 0
        )
        
        # --- 全连接分类层 ---
        # hidden_dim * 2: 因为是双向LSTM，前向和后向各有hidden_dim维
        self.fc = nn.Linear(hidden_dim * 2, num_classes)
        
        # --- Dropout ---
        self.dropout = nn.Dropout(0.5)
        
        # --- Sigmoid ---
        self.sigmoid = nn.Sigmoid()
    
    def forward(self, x):
        """
        前向传播
        
        数据流:
            x: (batch, seq_len)
            → embedding: (batch, seq_len, embed_dim)
            → BiLSTM: output (batch, seq_len, hidden*2), (h_n, c_n)
            → 取最后时间步: (batch, hidden*2)
            → dropout → fc → sigmoid: (batch,)
        
        LSTM输出说明:
            output: 每个时间步的隐状态 (batch, seq_len, hidden_dim*2)
            h_n:    最后时间步的隐状态 (num_layers*2, batch, hidden_dim)
            c_n:    最后时间步的记忆状态 (num_layers*2, batch, hidden_dim)
        """
        # Embedding
        embed = self.embedding(x)  # (batch, seq_len, embed_dim)
        
        # BiLSTM处理
        # lstm_out: 每个时间步的输出 (batch, seq_len, hidden_dim*2)
        # (h_n, c_n): 最终隐状态和记忆状态
        lstm_out, (h_n, c_n) = self.lstm(embed)
        
        # 取最后一个时间步的输出作为整个序列的表示
        # lstm_out[:, -1, :] 包含了前向最后时间步和后向第一个时间步的信息
        # 这是获取双向LSTM句子表示的常用方法
        out = lstm_out[:, -1, :]  # (batch, hidden_dim*2)
        
        # 分类
        out = self.dropout(out)
        out = self.fc(out)
        return self.sigmoid(out).squeeze(1)


# --- 初始化BiLSTM模型 ---
model_lstm = BiLSTM(
    vocab_size=VOCAB_SIZE,
    embed_dim=EMBEDDING_DIM,
    hidden_dim=64,
    num_layers=1
).to(DEVICE)

criterion_lstm = nn.BCELoss()
optimizer_lstm = optim.Adam(model_lstm.parameters(), lr=LEARNING_RATE)

# 打印模型结构
print("BiLSTM 模型结构:")
print(model_lstm)
total_params = sum(p.numel() for p in model_lstm.parameters())
print(f"\n总参数量: {total_params:,}")

# --- 训练BiLSTM ---
history_lstm = train_and_evaluate(
    model_lstm, "BiLSTM", train_loader, test_loader,
    criterion_lstm, optimizer_lstm, EPOCHS
)

# --- 最终评估 ---
lstm_acc, lstm_f1, lstm_probs, lstm_labels = eval_model(model_lstm, test_loader)
lstm_preds = (lstm_probs > 0.5).astype(int)
lstm_precision = precision_score(lstm_labels, lstm_preds)
lstm_recall = recall_score(lstm_labels, lstm_preds)

print(f"\n--- BiLSTM 最终评估结果 ---")
print(f"Accuracy:  {lstm_acc:.4f}")
print(f"Precision: {lstm_precision:.4f}")
print(f"Recall:    {lstm_recall:.4f}")
print(f"F1 Score:  {lstm_f1:.4f}")
print(classification_report(lstm_labels, lstm_preds, target_names=['负面', '正面']))

In [ ]:
# --- BiLSTM 训练过程可视化 ---
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 训练损失曲线
axes[0].plot(epochs_range, history_lstm['train_loss'], 'r-o', linewidth=2, markersize=6)
axes[0].set_title('BiLSTM 训练损失曲线', fontsize=14)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].grid(True, alpha=0.3)

# 测试准确率曲线
axes[1].plot(epochs_range, history_lstm['test_acc'], 'g-o', linewidth=2, markersize=6)
axes[1].set_title('BiLSTM 测试准确率曲线', fontsize=14)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].grid(True, alpha=0.3)

# 混淆矩阵
cm_lstm = confusion_matrix(lstm_labels, lstm_preds)
sns.heatmap(cm_lstm, annot=True, fmt='d', cmap='Oranges', ax=axes[2],
            xticklabels=['负面', '正面'], yticklabels=['负面', '正面'])
axes[2].set_title('BiLSTM 混淆矩阵', fontsize=14)
axes[2].set_xlabel('预测标签')
axes[2].set_ylabel('真实标签')

plt.tight_layout()
plt.savefig('bilstm_results.png', dpi=150, bbox_inches='tight')
plt.show()

## 第10部分：模型4 - BERT（预训练模型微调）

### 算法原理
BERT (Bidirectional Encoder Representations from Transformers) 由Google在2018年提出，
是NLP领域的里程碑式工作。

### 核心思想
- **预训练 + 微调 (Pre-train + Fine-tune)** 范式
  - 预训练阶段：在大规模无标注语料（如Wikipedia）上学习通用语言表示
  - 微调阶段：在特定下游任务上用少量标注数据微调
- **Transformer Encoder**: 基于自注意力机制（Self-Attention），能并行处理序列
- **双向上下文**: 同时利用左右两侧的上下文信息（不同于GPT的单向）
- **WordPiece分词**: 子词级别分词，能处理未登录词

### 微调策略
- 使用很小的学习率（2e-5），避免破坏预训练权重
- 通常只需2-3个epoch即可收敛
- 使用子集训练以加速演示（训练集5000条，测试集2000条）

### 网络结构
```
Input → BERT Tokenizer → [CLS] token1 token2 ... [SEP]
     → BERT Encoder (12层Transformer)
     → [CLS]对应的隐状态 → 分类头(Linear) → Softmax → 类别概率
```

In [ ]:
# ============================================================================
# 第10部分：BERT 模型微调
# ============================================================================
print("=" * 50)
print("训练模型4：BERT (bert-base-uncased)")
print("=" * 50)

# --- 10.1 加载BERT Tokenizer ---
# BERT使用WordPiece分词器，将文本拆分为子词单元
# 例如: "unbelievable" → ["un", "##believ", "##able"]
# 这样可以处理任何未登录词（OOV），因为任何词都可以拆分为已知的子词
tokenizer_bert = BertTokenizer.from_pretrained('bert-base-uncased')
print(f"BERT词表大小: {tokenizer_bert.vocab_size}")

# --- 10.2 BERT文本编码 ---
def encode_bert(texts, max_len=BERT_MAX_LEN):
    """
    使用BERT Tokenizer对文本进行编码
    
    BERT输入需要两个张量:
        - input_ids:      token的索引序列
        - attention_mask:  1=真实token, 0=padding（告诉模型哪些位置需要关注）
    
    BERT特殊token:
        - [CLS]: 句子开头，其对应的隐状态用于分类
        - [SEP]: 句子结尾/句子分隔符
        - [PAD]: 填充符
    
    示例:
        输入: "good movie"
        编码: [101, 2204, 3185, 102, 0, 0, ..., 0]
              [CLS] good  movie [SEP] [PAD] ...
        mask: [1,   1,    1,    1,   0, 0, ..., 0]
    
    参数:
        texts:   文本列表/Series
        max_len: 最大序列长度
    
    返回:
        input_ids:      (N, max_len) token索引张量
        attention_mask:  (N, max_len) 注意力掩码张量
    """
    encodings = tokenizer_bert(
        texts.tolist(),             # 转为Python列表
        truncation=True,            # 超过max_len则截断
        padding='max_length',       # 不足max_len则填充到max_length
        max_length=max_len,
        return_tensors='pt'         # 返回PyTorch张量
    )
    return encodings['input_ids'], encodings['attention_mask']


# --- 10.3 准备BERT数据 ---
# BERT模型参数量大（约1.1亿），训练较慢
# 为了在合理时间内完成实验，使用数据子集进行训练
BERT_TRAIN_SIZE = 5000   # 训练集取5000条
BERT_TEST_SIZE  = 2000   # 测试集取2000条

print(f"\nBERT使用数据子集: 训练{BERT_TRAIN_SIZE}条, 测试{BERT_TEST_SIZE}条")

# 取子集
train_texts_bert  = train_df['clean_text'][:BERT_TRAIN_SIZE]
test_texts_bert   = test_df['clean_text'][:BERT_TEST_SIZE]
train_labels_bert = torch.tensor(y_train[:BERT_TRAIN_SIZE], dtype=torch.long)
test_labels_bert  = torch.tensor(y_test[:BERT_TEST_SIZE], dtype=torch.long)

# BERT编码
print("BERT编码训练集...")
train_ids_bert, train_masks_bert = encode_bert(train_texts_bert)
print("BERT编码测试集...")
test_ids_bert, test_masks_bert = encode_bert(test_texts_bert)

print(f"BERT编码完成: train_ids={train_ids_bert.shape}, test_ids={test_ids_bert.shape}")

# 创建DataLoader
# BERT模型较大，batch_size需要设小一些（16），否则显存不够
train_dataset_bert = data.TensorDataset(train_ids_bert, train_masks_bert, train_labels_bert)
test_dataset_bert  = data.TensorDataset(test_ids_bert, test_masks_bert, test_labels_bert)
train_loader_bert  = data.DataLoader(train_dataset_bert, batch_size=16, shuffle=True)
test_loader_bert   = data.DataLoader(test_dataset_bert, batch_size=16, shuffle=False)

print(f"BERT DataLoader: 训练{len(train_loader_bert)}个batch, 测试{len(test_loader_bert)}个batch")

In [ ]:
# --- 10.4 加载预训练BERT模型 ---
# BertForSequenceClassification: BERT + 分类头
# 在BERT的[CLS] token输出上接一个线性分类层
# num_labels=2: 二分类（输出2个logits，经softmax后得到概率）
model_bert = BertForSequenceClassification.from_pretrained(
    'bert-base-uncased',
    num_labels=2
).to(DEVICE)

# AdamW: 带权重衰减(Weight Decay)的Adam优化器
# 权重衰减是一种正则化技术，防止参数值过大导致过拟合
# BERT微调推荐使用AdamW而非普通Adam
optimizer_bert = optim.AdamW(model_bert.parameters(), lr=BERT_LR)

total_params = sum(p.numel() for p in model_bert.parameters())
print(f"BERT 总参数量: {total_params:,} ({total_params/1e6:.1f}M)")


# --- 10.5 BERT训练函数 ---
def train_bert_epoch(epoch):
    """
    训练BERT一个epoch
    
    与普通模型的区别:
        1. 输入需要 input_ids 和 attention_mask 两个张量
        2. 传入labels参数，模型内部自动计算CrossEntropyLoss
        3. 输出是一个对象，包含 loss 和 logits
    """
    model_bert.train()
    total_loss = 0
    
    for batch_idx, (ids, masks, labels) in enumerate(train_loader_bert):
        # 将数据移到GPU
        ids    = ids.to(DEVICE)
        masks  = masks.to(DEVICE)
        labels = labels.to(DEVICE)
        
        optimizer_bert.zero_grad()
        
        # BERT前向传播
        # 传入labels后，模型会自动计算CrossEntropyLoss
        outputs = model_bert(
            input_ids=ids,
            attention_mask=masks,
            labels=labels
        )
        
        loss = outputs.loss  # 交叉熵损失
        loss.backward()      # 反向传播
        optimizer_bert.step() # 更新参数
        
        total_loss += loss.item()
    
    return total_loss / len(train_loader_bert)


def eval_bert():
    """
    评估BERT模型
    
    BERT输出logits（未经softmax的原始分数），需要用argmax取最大值对应的类别
    """
    model_bert.eval()
    all_preds, all_probs, all_labels = [], [], []
    
    with torch.no_grad():
        for ids, masks, labels in test_loader_bert:
            ids   = ids.to(DEVICE)
            masks = masks.to(DEVICE)
            
            outputs = model_bert(input_ids=ids, attention_mask=masks)
            logits = outputs.logits  # (batch, 2) 两个类别的原始分数
            
            # softmax将logits转为概率分布
            probs = torch.softmax(logits, dim=1)[:, 1].cpu().numpy()  # 正面类的概率
            preds = torch.argmax(logits, dim=1).cpu().numpy()  # 取概率最大的类别
            
            all_probs.extend(probs)
            all_preds.extend(preds)
            all_labels.extend(labels.numpy())
    
    acc = accuracy_score(all_labels, all_preds)
    f1  = f1_score(all_labels, all_preds)
    
    return acc, f1, np.array(all_preds), np.array(all_probs), np.array(all_labels)


# --- 10.6 训练BERT ---
history_bert = {'train_loss': [], 'test_acc': [], 'test_f1': []}

print(f"\n{'='*50}")
print(f"开始训练 BERT (共{BERT_EPOCHS}个epoch)")
print(f"{'='*50}")

for epoch in range(BERT_EPOCHS):
    start_time = time.time()
    
    train_loss = train_bert_epoch(epoch)
    acc, f1, _, _, _ = eval_bert()
    
    elapsed = time.time() - start_time
    
    history_bert['train_loss'].append(train_loss)
    history_bert['test_acc'].append(acc)
    history_bert['test_f1'].append(f1)
    
    print(f"Epoch {epoch+1}/{BERT_EPOCHS} - "
          f"Loss: {train_loss:.4f} - "
          f"Acc: {acc:.4f} - "
          f"F1: {f1:.4f} - "
          f"耗时: {elapsed:.1f}s")

# --- 10.7 最终评估 ---
bert_acc, bert_f1, bert_preds, bert_probs, bert_labels = eval_bert()
bert_precision = precision_score(bert_labels, bert_preds)
bert_recall = recall_score(bert_labels, bert_preds)

print(f"\n--- BERT 最终评估结果 ---")
print(f"Accuracy:  {bert_acc:.4f}")
print(f"Precision: {bert_precision:.4f}")
print(f"Recall:    {bert_recall:.4f}")
print(f"F1 Score:  {bert_f1:.4f}")
print(classification_report(bert_labels, bert_preds, target_names=['负面', '正面']))

In [ ]:
# --- BERT 训练过程可视化 ---
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

bert_epochs_range = range(1, BERT_EPOCHS + 1)

# 训练损失曲线
axes[0].plot(bert_epochs_range, history_bert['train_loss'], 'm-o', linewidth=2, markersize=6)
axes[0].set_title('BERT 训练损失曲线', fontsize=14)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].grid(True, alpha=0.3)

# 测试准确率曲线
axes[1].plot(bert_epochs_range, history_bert['test_acc'], 'g-o', linewidth=2, markersize=6)
axes[1].set_title('BERT 测试准确率曲线', fontsize=14)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].grid(True, alpha=0.3)

# 混淆矩阵
cm_bert = confusion_matrix(bert_labels, bert_preds)
sns.heatmap(cm_bert, annot=True, fmt='d', cmap='Purples', ax=axes[2],
            xticklabels=['负面', '正面'], yticklabels=['负面', '正面'])
axes[2].set_title('BERT 混淆矩阵', fontsize=14)
axes[2].set_xlabel('预测标签')
axes[2].set_ylabel('真实标签')

plt.tight_layout()
plt.savefig('bert_results.png', dpi=150, bbox_inches='tight')
plt.show()

## 第11部分：实验结果汇总与综合对比分析

将四个模型的实验结果进行全面对比，包括：
1. 各项指标数值对比表
2. 准确率与F1分数柱状图
3. 所有模型的ROC曲线对比
4. 训练损失曲线对比（深度学习模型）
5. 所有模型混淆矩阵汇总

In [ ]:
# ============================================================================
# 第11部分：实验结果汇总与综合对比
# ============================================================================
print("=" * 60)
print("实验结果汇总")
print("=" * 60)

# --- 11.1 结果汇总表 ---
results_df = pd.DataFrame({
    '模型':     ['Naive Bayes', 'TextCNN', 'BiLSTM', 'BERT'],
    '准确率':   [nb_acc, cnn_acc, lstm_acc, bert_acc],
    '精确率':   [nb_precision, cnn_precision, lstm_precision, bert_precision],
    '召回率':   [nb_recall, cnn_recall, lstm_recall, bert_recall],
    'F1分数':   [nb_f1, cnn_f1, lstm_f1, bert_f1],
})

# 格式化显示（保留4位小数）
print("\n各模型评价指标对比:")
print(results_df.to_string(index=False, float_format='%.4f'))

# 找出最佳模型
best_model_idx = results_df['F1分数'].idxmax()
print(f"\n最佳模型（按F1分数）: {results_df.loc[best_model_idx, '模型']} "
      f"(F1={results_df.loc[best_model_idx, 'F1分数']:.4f})")

In [ ]:
# --- 11.2 准确率与F1分数柱状图对比 ---
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

model_names = results_df['模型'].tolist()
colors = ['#3498db', '#e74c3c', '#2ecc71', '#9b59b6']  # 蓝、红、绿、紫

# 准确率对比
bars1 = axes[0].bar(model_names, results_df['准确率'], color=colors, edgecolor='black', alpha=0.85)
axes[0].set_title('各模型准确率对比', fontsize=16)
axes[0].set_ylabel('Accuracy', fontsize=12)
axes[0].set_ylim(0.7, 1.0)  # 从0.7开始，更好地展示差异
axes[0].grid(axis='y', alpha=0.3)
# 在柱子上方标注数值
for bar, val in zip(bars1, results_df['准确率']):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                 f'{val:.4f}', ha='center', va='bottom', fontsize=11, fontweight='bold')

# F1分数对比
bars2 = axes[1].bar(model_names, results_df['F1分数'], color=colors, edgecolor='black', alpha=0.85)
axes[1].set_title('各模型F1分数对比', fontsize=16)
axes[1].set_ylabel('F1 Score', fontsize=12)
axes[1].set_ylim(0.7, 1.0)
axes[1].grid(axis='y', alpha=0.3)
for bar, val in zip(bars2, results_df['F1分数']):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                 f'{val:.4f}', ha='center', va='bottom', fontsize=11, fontweight='bold')

plt.suptitle('四种模型性能对比', fontsize=18, y=1.02)
plt.tight_layout()
plt.savefig('model_comparison_bar.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# --- 11.3 四项指标雷达图对比 ---
# 雷达图可以直观展示每个模型在多个维度上的表现

categories = ['准确率', '精确率', '召回率', 'F1分数']
N = len(categories)

# 计算角度
angles = [n / float(N) * 2 * np.pi for n in range(N)]
angles += angles[:1]  # 闭合

fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))

for i, model_name in enumerate(model_names):
    values = results_df.iloc[i, 1:].values.tolist()
    values += values[:1]  # 闭合
    ax.plot(angles, values, 'o-', linewidth=2, label=model_name, color=colors[i])
    ax.fill(angles, values, alpha=0.1, color=colors[i])

ax.set_xticks(angles[:-1])
ax.set_xticklabels(categories, fontsize=13)
ax.set_ylim(0.7, 1.0)
ax.set_title('四种模型多维度性能雷达图', fontsize=16, pad=20)
ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1), fontsize=11)
ax.grid(True)

plt.tight_layout()
plt.savefig('model_radar_chart.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# --- 11.4 ROC曲线对比 ---
# ROC曲线 (Receiver Operating Characteristic):
#   - X轴: 假正率 FPR = FP / (FP + TN)  （把负面错判为正面的比例）
#   - Y轴: 真正率 TPR = TP / (TP + FN)  （正确识别正面的比例）
#   - 曲线越靠近左上角，模型越好
#   - AUC (Area Under Curve): 曲线下面积，越接近1越好

plt.figure(figsize=(8, 7))

# Naive Bayes ROC
fpr_nb, tpr_nb, _ = roc_curve(y_test, y_prob_nb)
auc_nb = auc(fpr_nb, tpr_nb)
plt.plot(fpr_nb, tpr_nb, color=colors[0], linewidth=2,
         label=f'Naive Bayes (AUC={auc_nb:.4f})')

# TextCNN ROC
fpr_cnn, tpr_cnn, _ = roc_curve(cnn_labels, cnn_probs)
auc_cnn = auc(fpr_cnn, tpr_cnn)
plt.plot(fpr_cnn, tpr_cnn, color=colors[1], linewidth=2,
         label=f'TextCNN (AUC={auc_cnn:.4f})')

# BiLSTM ROC
fpr_lstm, tpr_lstm, _ = roc_curve(lstm_labels, lstm_probs)
auc_lstm = auc(fpr_lstm, tpr_lstm)
plt.plot(fpr_lstm, tpr_lstm, color=colors[2], linewidth=2,
         label=f'BiLSTM (AUC={auc_lstm:.4f})')

# BERT ROC
fpr_bert, tpr_bert, _ = roc_curve(bert_labels, bert_probs)
auc_bert = auc(fpr_bert, tpr_bert)
plt.plot(fpr_bert, tpr_bert, color=colors[3], linewidth=2,
         label=f'BERT (AUC={auc_bert:.4f})')

# 对角线（随机分类器基准线）
plt.plot([0, 1], [0, 1], 'k--', linewidth=1, label='随机分类器 (AUC=0.5)')

plt.xlabel('假正率 (False Positive Rate)', fontsize=12)
plt.ylabel('真正率 (True Positive Rate)', fontsize=12)
plt.title('四种模型 ROC 曲线对比', fontsize=16)
plt.legend(loc='lower right', fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('roc_curves_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\nAUC值汇总:")
print(f"  Naive Bayes: {auc_nb:.4f}")
print(f"  TextCNN:     {auc_cnn:.4f}")
print(f"  BiLSTM:      {auc_lstm:.4f}")
print(f"  BERT:        {auc_bert:.4f}")

In [ ]:
# --- 11.5 深度学习模型训练损失曲线对比 ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 训练损失对比
axes[0].plot(range(1, EPOCHS+1), history_cnn['train_loss'], 'r-o', 
             linewidth=2, markersize=6, label='TextCNN')
axes[0].plot(range(1, EPOCHS+1), history_lstm['train_loss'], 'g-s', 
             linewidth=2, markersize=6, label='BiLSTM')
axes[0].plot(range(1, BERT_EPOCHS+1), history_bert['train_loss'], 'm-^', 
             linewidth=2, markersize=6, label='BERT')
axes[0].set_title('深度学习模型训练损失对比', fontsize=14)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Training Loss')
axes[0].legend(fontsize=11)
axes[0].grid(True, alpha=0.3)

# 测试准确率对比
axes[1].plot(range(1, EPOCHS+1), history_cnn['test_acc'], 'r-o', 
             linewidth=2, markersize=6, label='TextCNN')
axes[1].plot(range(1, EPOCHS+1), history_lstm['test_acc'], 'g-s', 
             linewidth=2, markersize=6, label='BiLSTM')
axes[1].plot(range(1, BERT_EPOCHS+1), history_bert['test_acc'], 'm-^', 
             linewidth=2, markersize=6, label='BERT')
axes[1].set_title('深度学习模型测试准确率对比', fontsize=14)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Test Accuracy')
axes[1].legend(fontsize=11)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('dl_training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# --- 11.6 四个模型混淆矩阵汇总 ---
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

cms = [
    (cm_nb,   'Naive Bayes', 'Blues'),
    (cm_cnn,  'TextCNN',     'Greens'),
    (cm_lstm, 'BiLSTM',      'Oranges'),
    (cm_bert, 'BERT',        'Purples')
]

for ax, (cm, name, cmap) in zip(axes.flatten(), cms):
    sns.heatmap(cm, annot=True, fmt='d', cmap=cmap, ax=ax,
                xticklabels=['负面', '正面'], yticklabels=['负面', '正面'],
                annot_kws={'size': 14})
    ax.set_title(f'{name} 混淆矩阵', fontsize=14)
    ax.set_xlabel('预测标签', fontsize=11)
    ax.set_ylabel('真实标签', fontsize=11)

plt.suptitle('四种模型混淆矩阵汇总', fontsize=18, y=1.02)
plt.tight_layout()
plt.savefig('all_confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()

## 第12部分：实验结论与分析

### 实验总结

本实验在IMDB电影评论数据集上，系统性地对比了四种不同层次的文本情感分类方法：

| 模型 | 方法类别 | 特点 |
|------|----------|------|
| Naive Bayes + TF-IDF | 传统机器学习 | 简单高效，可解释性强，适合作为基准模型 |
| TextCNN | 深度学习（CNN） | 并行计算快，擅长捕捉局部n-gram特征 |
| BiLSTM | 深度学习（RNN） | 能捕捉长距离依赖和双向上下文信息 |
| BERT | 预训练模型微调 | 利用大规模预训练知识，语义理解能力最强 |

### 关键发现

1. **传统方法仍有竞争力**: Naive Bayes + TF-IDF 作为基准模型，在情感二分类任务上已能取得不错的效果，且训练速度极快。

2. **深度学习的优势**: TextCNN和BiLSTM通过学习词向量表示和上下文特征，通常能超越传统方法。

3. **预训练模型的威力**: BERT即使只使用部分训练数据，也能取得有竞争力的结果，体现了预训练+微调范式的优势。

4. **模型复杂度与性能的权衡**: 
   - 模型越复杂，参数量越大，训练时间越长
   - 但性能提升并非线性增长
   - 实际应用中需要根据资源和需求选择合适的模型

### 改进方向
- 使用更大的BERT训练数据量
- 尝试其他预训练模型（如RoBERTa、ALBERT）
- 集成学习（Ensemble）融合多个模型的预测
- 数据增强（回译、同义词替换等）

In [ ]:
# ============================================================================
# 第12部分：最终结果输出
# ============================================================================

# --- 完整结果汇总表（含AUC） ---
final_results = pd.DataFrame({
    '模型':     ['Naive Bayes', 'TextCNN', 'BiLSTM', 'BERT'],
    '准确率':   [nb_acc, cnn_acc, lstm_acc, bert_acc],
    '精确率':   [nb_precision, cnn_precision, lstm_precision, bert_precision],
    '召回率':   [nb_recall, cnn_recall, lstm_recall, bert_recall],
    'F1分数':   [nb_f1, cnn_f1, lstm_f1, bert_f1],
    'AUC':     [auc_nb, auc_cnn, auc_lstm, auc_bert],
})

print("=" * 70)
print("最终实验结果汇总")
print("=" * 70)
print(final_results.to_string(index=False, float_format='%.4f'))
print("=" * 70)
print("\n实验完成！所有图表已保存到当前目录。")